# Co-authorship Social Network Analysis

Reads the xlsx files produced by `build_network.py` and generates two
network figures per dataset: one labelled with author names, one with
node attributes instead.

Each figure uses Louvain community detection to colour clusters and a
community-aware force-directed layout to spatially separate them.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# xlsx files produced by build_network.py
# update the names here if your files differ
NETWORK_FILES = {
    "sample": "sample_network.xlsx"
    
}

SEED = 42   # keeps layouts identical across runs

In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────

def load_data(file_path):
    xls = pd.ExcelFile(file_path)
    nodes_sheet = "Nodes" if "Nodes" in xls.sheet_names else xls.sheet_names[0]
    edges_sheet = "Edges" if "Edges" in xls.sheet_names else xls.sheet_names[1]

    nodes = pd.read_excel(file_path, sheet_name=nodes_sheet)
    edges = pd.read_excel(file_path, sheet_name=edges_sheet)
    print(f"Loaded {len(nodes)} nodes and {len(edges)} edges from {file_path}")
    return nodes, edges


# ── Graph construction ────────────────────────────────────────────────────────

def build_graph(nodes, edges):
    G = nx.Graph()

    # Prefer the 'Author' column as node id; fall back to the first column
    id_col = "Author" if "Author" in nodes.columns else nodes.columns[0]

    for _, row in nodes.iterrows():
        attrs = {col: (None if pd.isna(row[col]) else row[col])
                 for col in nodes.columns}
        G.add_node(row[id_col], **attrs)

    src_col = edges.columns[0]
    tgt_col = edges.columns[1]
    weight_col = "weight" if "weight" in edges.columns else None

    # Accumulate weights for parallel edges before adding to the graph
    edge_weights = {}
    for _, row in edges.iterrows():
        src, tgt = row[src_col], row[tgt_col]
        if src not in G.nodes or tgt not in G.nodes:
            continue
        key = tuple(sorted([src, tgt]))
        w   = float(row[weight_col]) if weight_col and not pd.isna(row[weight_col]) else 1
        edge_weights[key] = edge_weights.get(key, 0) + w

    for (src, tgt), w in edge_weights.items():
        G.add_edge(src, tgt, weight=w)

    # Degree-based node attributes used later for sizing
    for node in G.nodes():
        G.nodes[node]["degree"] = G.degree(node)
        G.nodes[node]["weighted_degree"] = sum(
            G[node][nb].get("weight", 1) for nb in G.neighbors(node)
        )

    # Drop isolated nodes or near-isolates — too little signal for SNA
    isolates = [n for n, d in G.degree() if d <= 2]
    G.remove_nodes_from(isolates)

    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges "
          f"(removed {len(isolates)} low-degree nodes)")
    return G


# ── Community detection ───────────────────────────────────────────────────────
# Uses the Louvain algorithm (python-louvain package). Falls back to NetworkX's
# greedy modularity method if Louvain isn't installed.
# Communities are renumbered 1..N after detection so colours stay stable.

def detect_communities(G, seed=42):
    try:
        try:
            import community.community_louvain as cl
        except ImportError:
            import community as cl

        import inspect
        if "random_state" in inspect.signature(cl.best_partition).parameters:
            partition = cl.best_partition(G, random_state=seed)
        else:
            np.random.seed(seed)
            partition = cl.best_partition(G)
        print("Community detection: Louvain")

    except ImportError:
        # Louvain not installed — greedy modularity is deterministic enough here
        import networkx.algorithms.community as nx_comm
        np.random.seed(seed)
        partition = {}
        for i, comm in enumerate(nx_comm.greedy_modularity_communities(G)):
            for node in comm:
                partition[node] = i
        print("Community detection: NetworkX greedy modularity (fallback)")

    nx.set_node_attributes(G, partition, "community")

    # Renumber so community IDs are always 1, 2, 3 … regardless of algorithm output
    old_to_new = {old: i + 1 for i, old in enumerate(sorted(set(partition.values())))}
    for node in G.nodes():
        G.nodes[node]["community"] = old_to_new[G.nodes[node]["community"]]

    counts = {}
    for node in G.nodes():
        c = G.nodes[node]["community"]
        counts[c] = counts.get(c, 0) + 1

    print(f"Detected {len(counts)} communities")
    return sorted(counts.keys())


def filter_small_communities(G, min_size=2):
    """Drop communities with fewer than min_size members — not meaningful for SNA."""
    counts = {}
    for node in G.nodes():
        c = G.nodes[node]["community"]
        counts[c] = counts.get(c, 0) + 1

    small = {c for c, n in counts.items() if n < min_size}
    to_remove = [node for node in G.nodes() if G.nodes[node]["community"] in small]
    G.remove_nodes_from(to_remove)
    print(f"Removed {len(to_remove)} nodes from {len(small)} small communities")
    return G


# ── Layout ────────────────────────────────────────────────────────────────────
# Two-level layout: first position the communities relative to each other,
# then lay out each community's internal nodes. This keeps communities
# visually separate while showing internal structure.

def community_layout(G, seed=42):
    community_dict = nx.get_node_attributes(G, "community")
    comm_to_nodes  = {}
    for node, c in community_dict.items():
        comm_to_nodes.setdefault(c, []).append(node)

    # Meta-graph: one node per community, edges where communities share connections
    meta = nx.Graph()
    for c in comm_to_nodes:
        meta.add_node(c)
    for u, v in G.edges():
        cu, cv = community_dict[u], community_dict[v]
        if cu != cv:
            if meta.has_edge(cu, cv):
                meta[cu][cv]["weight"] += 1
            else:
                meta.add_edge(cu, cv, weight=1)

    # Circular layout is deterministic and works well for ≤10 communities
    if len(comm_to_nodes) <= 10:
        meta_pos = {c: (x * 6, y * 6)
                    for c, (x, y) in nx.circular_layout(meta).items()}
    else:
        raw = nx.spring_layout(meta, k=8.0, iterations=150, seed=seed, weight="weight")
        meta_pos = {c: (x * 6, y * 6) for c, (x, y) in raw.items()}

    pos = {}
    for c, nodes in comm_to_nodes.items():
        subgraph = G.subgraph(nodes)
        if len(nodes) <= 5:
            sub_pos = nx.circular_layout(subgraph)
        else:
            sub_pos = nx.spring_layout(
                subgraph,
                k=1.5 / np.sqrt(len(nodes)),
                iterations=200,
                seed=c * seed   # each community gets its own deterministic seed
            )

        cx, cy = meta_pos[c]
        scale  = 0.3 + 0.08 * np.log1p(len(nodes))
        for node, (x, y) in sub_pos.items():
            pos[node] = (cx + x * scale, cy + y * scale)

    return pos


def apply_repulsion(pos, G, iterations=50, strength=0.05, seed=42):
    """Nudge overlapping nodes apart, respecting community boundaries."""
    np.random.seed(seed)
    community_dict = nx.get_node_attributes(G, "community")
    node_list = sorted(G.nodes())   # consistent order for reproducibility
    new_pos = dict(pos)

    for _ in range(iterations):
        disp = {n: np.zeros(2) for n in node_list}
        for i, u in enumerate(node_list):
            for v in node_list[i + 1:]:
                delta = np.array(new_pos[v]) - np.array(new_pos[u])
                dist  = max(0.01, np.linalg.norm(delta))
                # inter-community pairs repel more strongly to maintain separation
                scale = 0.5 if community_dict[u] == community_dict[v] else 2.0
                force = delta / dist * strength * scale / dist ** 2
                disp[u] -= force
                disp[v] += force

        for node in node_list:
            damping = 0.3 + G.nodes[node].get("degree", 1) / 30
            new_pos[node] = (
                new_pos[node][0] + disp[node][0] * damping,
                new_pos[node][1] + disp[node][1] * damping,
            )

    return new_pos


# ── Visualisation prep ────────────────────────────────────────────────────────
# Assigns colours, flags the top-3 highest-degree nodes per community for
# labelling, and computes the final node positions.

def prepare_visualization(G, seed=42):
    present_comms = sorted(set(nx.get_node_attributes(G, "community").values()))

    # Colorblind-friendly palette; cycle through tab20 if we need more than 15
    palette = [
        "#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00",
        "#FFFF33", "#A65628", "#F781BF", "#999999", "#66C2A5",
        "#FC8D62", "#8DA0CB", "#E78AC3", "#A6D854", "#FFD92F",
    ]
    if len(present_comms) <= len(palette):
        colors = palette[:len(present_comms)]
    else:
        cmap   = plt.colormaps["tab20"]
        colors = [mcolors.rgb2hex(cmap(i / len(present_comms)))
                  for i in range(len(present_comms))]

    color_map = dict(zip(present_comms, colors))

    for node in G.nodes():
        G.nodes[node]["node_color"]  = color_map[G.nodes[node]["community"]]
        G.nodes[node]["is_top_node"] = False

    # Label the 3 most-connected authors in each community
    for c in present_comms:
        members = [n for n in G.nodes() if G.nodes[n]["community"] == c]
        if len(members) < 2:
            continue
        top3 = sorted(members,
                      key=lambda n: (-G.nodes[n]["weighted_degree"], n))[:3]
        for n in top3:
            G.nodes[n]["is_top_node"] = True

    pos = community_layout(G, seed=seed)
    pos = apply_repulsion(pos, G, iterations=150, strength=0.04, seed=seed)
    return pos, color_map


# ── Drawing ───────────────────────────────────────────────────────────────────

def draw_network(G, pos, color_map, show_names=True, base_size=500, output_file=None):
    fig, ax = plt.subplots(figsize=(14, 12))

    present_comms = sorted(set(nx.get_node_attributes(G, "community").values()))

    # --- Edges ---
    for u, v in sorted(G.edges()):
        if u not in pos or v not in pos:
            continue
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        cu = G.nodes[u].get("community", 0)
        cv = G.nodes[v].get("community", 0)
        raw_w = G[u][v].get("weight", 1)
        w = np.sqrt(min(10, max(1, float(raw_w))))  # sqrt keeps thick edges readable

        if cu == cv:
            # Intra-community: dashed, coloured — makes cluster structure obvious
            ax.plot([x1, x2], [y1, y2],
                    color=color_map.get(cu, "gray"), alpha=0.75,
                    linewidth=1.2 * w, linestyle="--", dashes=(4, 2))
        else:
            # Inter-community: faint curved line so cross-cluster ties are visible
            # but don't clutter the figure
            ax.annotate("",
                        xy=(x2, y2), xycoords="data",
                        xytext=(x1, y1), textcoords="data",
                        arrowprops=dict(arrowstyle="-", color="black",
                                        alpha=0.3, linewidth=0.6 * w,
                                        connectionstyle="arc3,rad=0.15"))

    # --- Nodes ---
    for c in present_comms:
        members = sorted([n for n in G.nodes() if G.nodes[n]["community"] == c])
        if not members:
            continue

        wds  = [G.nodes[n]["weighted_degree"] for n in members]
        mn, mx = min(wds), max(wds)
        rng  = max(1, mx - mn)

        def node_size(n):
            return base_size * (0.7 + 1.3 * (G.nodes[n]["weighted_degree"] - mn) / rng)

        regular = [n for n in members if not G.nodes[n]["is_top_node"]]
        top     = [n for n in members if G.nodes[n]["is_top_node"]]
        color   = color_map.get(c, "gray")

        if regular:
            ax.scatter([pos[n][0] for n in regular], [pos[n][1] for n in regular],
                       s=[node_size(n) for n in regular], c=color,
                       edgecolors="black", linewidths=0.5, alpha=0.8, zorder=10)

        if top:
            ax.scatter([pos[n][0] for n in top], [pos[n][1] for n in top],
                       s=[node_size(n) * 1.5 for n in top], c=color,
                       edgecolors="black", linewidths=1.2, alpha=0.9, zorder=11)

            for n in top:
                if show_names:
                    label = str(n)
                else:
                    # Show the first non-system attribute value available
                    skip = {"community", "node_color", "is_top_node",
                            "degree", "weighted_degree", "Author"}
                    extras = {k: v for k, v in G.nodes[n].items()
                              if k not in skip and v is not None
                              and not (isinstance(v, float) and np.isnan(v))}
                    if extras:
                        k = sorted(extras)[0]
                        v = extras[k]
                        label = f"{v:.2f}" if isinstance(v, float) else str(v)
                    else:
                        label = f"WD: {G.nodes[n]['weighted_degree']:.0f}"

                ax.text(pos[n][0], pos[n][1] + 0.02, label,
                        fontsize=9, ha="center", va="center", color="blue",
                        fontweight="bold", zorder=12,
                        bbox=dict(boxstyle="round,pad=0.3", fc="white",
                                  ec="black", alpha=0.9))

    # --- Legend ---
    legend = [plt.Line2D([0], [0], marker="o", color="w",
                          markerfacecolor=color_map.get(c, "gray"),
                          markersize=10, label=f"Community {c}")
              for c in present_comms[:10]]   # cap at 10 to keep legend tidy
    if len(present_comms) > 10:
        legend.append(plt.Line2D([0], [0], marker="o", color="w",
                                  markerfacecolor="gray", markersize=10,
                                  label="Other communities"))
    legend += [
        plt.Line2D([0], [0], color="gray", lw=1.2, linestyle="--", dashes=(4, 2),
                   label="Intra-community edge"),
        plt.Line2D([0], [0], color="black", lw=0.6, alpha=0.5,
                   label="Inter-community edge"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="gray",
                   markeredgecolor="black", markeredgewidth=1.5, markersize=12,
                   label="Top node (by weighted degree)"),
    ]
    ax.legend(handles=legend, loc="upper center", bbox_to_anchor=(0.5, -0.05),
              ncol=min(3, len(legend)), frameon=True, fontsize=9)

    title = "With Author Names" if show_names else "With Node Attributes"
    ax.set_title(f"Co-authorship Network — {title}", fontsize=16)
    ax.axis("off")
    plt.tight_layout(pad=2.0)

    if output_file:
        plt.savefig(output_file, dpi=300, bbox_inches="tight")
        print(f"Saved: {output_file}")
    plt.show()


# ── Main pipeline ─────────────────────────────────────────────────────────────

def visualize_network(file_path, label="network", seed=42):
    np.random.seed(seed)

    nodes, edges = load_data(file_path)
    G = build_graph(nodes, edges)

    detect_communities(G, seed=seed)
    G = filter_small_communities(G, min_size=2)

    if G.number_of_nodes() == 0:
        print("No nodes left after filtering.")
        return

    pos, color_map = prepare_visualization(G, seed=seed)

    print("\nPlotting with author names...")
    draw_network(G, pos, color_map, show_names=True,
                 output_file=f"{label}_names.png")

    print("Plotting with node attributes...")
    draw_network(G, pos, color_map, show_names=False,
                 output_file=f"{label}_attributes.png")

In [ ]:
visualize_network(NETWORK_FILES["sample"], label="sample", seed=SEED)